# Mini pLTV prototype

**Step 1:** data loading, audit and cleaning · **Step 2:** features and target · **Step 3:** pLTV model vs. naive signal · **Step 4:** experiment design and power

**Project question:** does a customer's first order tell us how much they will be worth over time?

**Dataset:** *Online Retail II* (UCI Machine Learning Repository). Real transactions of a UK online gift retailer, Dec 2009 – Dec 2011. One row = one product line on an invoice.

**Goal of this step:** turn raw, messy transaction data into a clean, customer-level event table we can trust for modelling. Every cleaning decision is logged with the number of rows it affects and the reason, so the process is auditable.

> This mirrors the first job of a client data integration: connect to raw data, find what's broken, and map it into a common data model.

## 1.0 Setup
Libraries: `pandas` for tables, `numpy` for numerics. Display options make wide tables readable.

In [1]:
import os, zipfile, urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1.1 Download the data
Downloads the zip from UCI once (skipped if already present) and extracts the Excel file.

In [2]:
URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
ZIP_PATH = "online_retail_ii.zip"

if not os.path.exists(ZIP_PATH):
    urllib.request.urlretrieve(URL, ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH) as z:
    print(z.namelist())
    z.extractall(".")

XLSX_PATH = [f for f in os.listdir(".") if f.lower().endswith(".xlsx")][0]
print("Using:", XLSX_PATH)

['online_retail_II.xlsx']
Using: online_retail_II.xlsx


## 1.2 Load both sheets and combine them
The Excel file has **two sheets** (one per year). `sheet_name=None` loads all sheets into a dictionary.

`Invoice` and `StockCode` are forced to text: some invoices start with a letter (e.g. `C` for cancellations) and some product codes contain letters; read as numbers they would be corrupted.

Columns are renamed to `snake_case` so the code is consistent and easy to type. *(Reading the Excel file takes 1–2 minutes.)*

In [3]:
sheets = pd.read_excel(XLSX_PATH, sheet_name=None, dtype={"Invoice": str, "StockCode": str})
for name, df in sheets.items():
    print(f"{name}: {df.shape[0]:,} rows | {df['InvoiceDate'].min()} -> {df['InvoiceDate'].max()}")

raw = pd.concat(sheets.values(), ignore_index=True)
raw = raw.rename(columns={
    "Invoice": "invoice", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "invoice_date", "Price": "price",
    "Customer ID": "customer_id", "Country": "country",
})
print(f"\nCombined: {raw.shape[0]:,} rows, {raw.shape[1]} columns")

Year 2009-2010: 525,461 rows | 2009-12-01 07:45:00 -> 2010-12-09 20:01:00
Year 2010-2011: 541,910 rows | 2010-12-01 08:26:00 -> 2011-12-09 12:50:00

Combined: 1,067,371 rows, 8 columns


## 1.3 First look
Before changing anything: column types, a few rows, and summary statistics. We are looking for anything suspicious — negative quantities, zero prices, missing values.

In [4]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice       1067371 non-null  object        
 1   stock_code    1067371 non-null  object        
 2   description   1062989 non-null  object        
 3   quantity      1067371 non-null  int64         
 4   invoice_date  1067371 non-null  datetime64[ns]
 5   price         1067371 non-null  float64       
 6   customer_id   824364 non-null   float64       
 7   country       1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [5]:
raw.head(10)

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,"13,085.00",United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,"13,085.00",United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,"13,085.00",United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,"13,085.00",United Kingdom


In [6]:
raw[["quantity", "price"]].describe(percentiles=[0.01, 0.5, 0.99])

,quantity,price
count,"1,067,371.00","1,067,371.00"
mean,9.94,4.65
std,172.71,123.55
min,"-80,995.00","-53,594.36"
1%,-3.00,0.21
50%,3.00,2.10
99%,100.00,18.00
max,"80,995.00","38,970.00"


## 1.4 Data quality audit
We **measure** every problem before fixing any of it. Each check is a boolean mask (True = the row has the problem), so we can count it and reuse it.

| Check | Why it matters for LTV |
|---|---|
| Missing customer ID | Without an ID we can't link purchases to a person, so no LTV can be computed |
| Exact duplicates | Same row twice = revenue counted twice |
| Cancellations (`C` invoices) | These are **returns**: they reduce a customer's real value |
| Negative quantity, not a cancellation | Stock adjustments, not customer behaviour |
| Price ≤ 0 | Free samples, write-offs, bad-debt adjustments |
| Non-product codes | Postage, fees, manual adjustments: not purchases |

In [7]:
is_cancel = raw["invoice"].str.startswith("C")
is_product = raw["stock_code"].str.match(r"^\d{5}", na=False)

checks = {
    "missing_customer_id": raw["customer_id"].isna(),
    "exact_duplicate": raw.duplicated(),
    "cancellation_invoice": is_cancel,
    "negative_qty_not_cancellation": (raw["quantity"] < 0) & ~is_cancel,
    "price_le_zero": raw["price"] <= 0,
    "non_product_code": ~is_product,
}

audit = pd.DataFrame({
    "rows": {k: int(v.sum()) for k, v in checks.items()},
    "pct_of_total": {k: 100 * v.mean() for k, v in checks.items()},
})
audit

,rows,pct_of_total
missing_customer_id,243007,22.77
exact_duplicate,34335,3.22
cancellation_invoice,19494,1.83
negative_qty_not_cancellation,3457,0.32
price_le_zero,6207,0.58
non_product_code,6093,0.57


What are the non-product codes? Listing them confirms the regex catches fees and adjustments, not real products.

In [8]:
raw.loc[~is_product, ["stock_code", "description"]].value_counts().head(20)

,,count
stock_code,description,
POST,POSTAGE,2115
DOT,DOTCOM POSTAGE,1444
M,Manual,1421
C2,CARRIAGE,279
D,Discount,177
S,SAMPLES,104
BANK CHARGES,Bank Charges,96
AMAZONFEE,AMAZON FEE,43
ADJUST,Adjustment by john on 26/01/2010 16,38


**Overlap between sheets.** If the two yearly sheets share some days, the same transactions appear twice. We check the date range shared by both.

In [9]:
ranges = {n: (d["InvoiceDate"].min(), d["InvoiceDate"].max()) for n, d in sheets.items()}
(n1, (s1, e1)), (n2, (s2, e2)) = list(ranges.items())[:2]
overlap_start, overlap_end = max(s1, s2), min(e1, e2)
in_overlap = raw["invoice_date"].between(overlap_start, overlap_end)
print(f"Overlap window: {overlap_start} -> {overlap_end}")
print(f"Rows in overlap: {in_overlap.sum():,} | of which exact duplicates: {(in_overlap & raw.duplicated()).sum():,}")

Overlap window: 2010-12-01 08:26:00 -> 2010-12-09 20:01:00
Rows in overlap: 45,046 | of which exact duplicates: 22,844


## 1.5 Cleaning, step by step (with a log)
Each step records rows before/after and **why** we did it. The order matters: duplicates first (so later counts are honest), then rows we can't use.

**Key decision — returns are kept, not deleted.** Many tutorials drop cancellations. For LTV that would be wrong: a customer who buys €200 and returns €150 is worth €50. We split them into a separate returns table.

In [10]:
log = []

def apply_step(df, mask_keep, name, why):
    out = df[mask_keep].copy()
    log.append({"step": name, "rows_before": len(df), "rows_after": len(out),
                "removed": len(df) - len(out), "why": why})
    return out

df = raw.copy()
df = apply_step(df, ~df.duplicated(), "drop exact duplicates",
                "same transaction recorded twice (incl. sheet overlap)")
df = apply_step(df, df["customer_id"].notna(), "drop missing customer_id",
                "cannot attribute revenue to a customer -> no LTV")
df = apply_step(df, df["stock_code"].str.match(r"^\d{5}", na=False), "drop non-product codes",
                "postage, fees, manual adjustments are not purchases")
df = apply_step(df, df["price"] > 0, "drop price <= 0",
                "free items, write-offs, bad-debt adjustments")

is_ret = df["invoice"].str.startswith("C")
df = apply_step(df, is_ret | (df["quantity"] > 0), "drop negative qty outside cancellations",
                "stock corrections, not customer behaviour")

pd.DataFrame(log)

,step,rows_before,rows_after,removed,why
0,drop exact duplicates,1067371,1033036,34335,same transaction recorded twice (incl. sheet o...
1,drop missing customer_id,1033036,797885,235151,cannot attribute revenue to a customer -> no LTV
2,drop non-product codes,797885,794223,3662,"postage, fees, manual adjustments are not purc..."
3,drop price <= 0,794223,794163,60,"free items, write-offs, bad-debt adjustments"
4,drop negative qty outside cancellations,794163,794163,0,"stock corrections, not customer behaviour"


## 1.6 Types, revenue, and the purchase / return split
- `customer_id` becomes text: it's an identifier, not a number we do maths on (1.5 → `"12346"`).
- `revenue = quantity × price`. Returns have negative quantity, so their revenue is negative automatically.
- `event_type` labels each line as `purchase` or `return`.

In [11]:
df["customer_id"] = df["customer_id"].astype(int).astype(str)
df["revenue"] = df["quantity"] * df["price"]
df["event_type"] = np.where(df["invoice"].str.startswith("C"), "return", "purchase")

df.groupby("event_type").agg(lines=("invoice", "size"),
                             invoices=("invoice", "nunique"),
                             revenue=("revenue", "sum"))

,lines,invoices,revenue
event_type,,,
purchase,776577,36594,"17,068,567.97"
return,17586,7282,"-709,953.43"


## 1.7 Build the event table (our "common data model")
Line-level data (one row per product) is too granular for LTV. We aggregate to **one row per invoice**: who, when, what type of event, how much. This is the standard shape a pLTV pipeline consumes: `customer_id, event_time, event_type, value`.

In [12]:
events = (df.groupby(["customer_id", "invoice", "event_type"], as_index=False)
            .agg(event_time=("invoice_date", "min"),
                 value=("revenue", "sum"),
                 n_items=("quantity", "sum"),
                 n_lines=("stock_code", "size"),
                 country=("country", "first"))
            .sort_values(["customer_id", "event_time"])
            .reset_index(drop=True))
events.head(10)

,customer_id,invoice,event_type,event_time,value,n_items,n_lines,country
0,12346,499763,purchase,2010-03-02 13:08:00,27.05,5,5,United Kingdom
1,12346,513774,purchase,2010-06-28 13:53:00,142.31,19,19,United Kingdom
2,12346,541431,purchase,2011-01-18 10:01:00,"77,183.60",74215,1,United Kingdom
3,12346,C541433,return,2011-01-18 10:17:00,"-77,183.60",-74215,1,United Kingdom
4,12347,529924,purchase,2010-10-31 14:20:00,611.53,509,40,Iceland
5,12347,537626,purchase,2010-12-07 14:57:00,711.79,319,31,Iceland
6,12347,542237,purchase,2011-01-26 14:30:00,475.39,315,29,Iceland
7,12347,549222,purchase,2011-04-07 10:43:00,636.25,483,24,Iceland
8,12347,556201,purchase,2011-06-09 13:01:00,382.52,196,18,Iceland
9,12347,562032,purchase,2011-08-02 08:48:00,584.91,277,22,Iceland


**Each customer's order sequence.** For every purchase we compute, *per customer*:
- `order_nr`: 1 for the first purchase, 2 for the second, and so on (`cumcount` counts rows within each customer, starting at 0, so we add 1);
- `days_since_prev_order`: days since that customer's previous purchase (`diff` subtracts the previous row's date within the same customer).

`groupby("customer_id")` makes both calculations restart for every customer. The gap between the 1st and 2nd purchase is one of the strongest early signals of long-term value — we'll use it in step 2.

In [13]:
orders_seq = (events[events["event_type"] == "purchase"]
              [["customer_id", "invoice", "event_time", "value"]]
              .sort_values(["customer_id", "event_time"])
              .reset_index(drop=True))

orders_seq["order_nr"] = orders_seq.groupby("customer_id").cumcount() + 1
orders_seq["days_since_prev_order"] = (orders_seq.groupby("customer_id")["event_time"]
                                       .diff().dt.days)
orders_seq.head(10)

,customer_id,invoice,event_time,value,order_nr,days_since_prev_order
0,12346,499763,2010-03-02 13:08:00,27.05,1,NaN
1,12346,513774,2010-06-28 13:53:00,142.31,2,118.00
2,12346,541431,2011-01-18 10:01:00,"77,183.60",3,203.00
3,12347,529924,2010-10-31 14:20:00,611.53,1,NaN
4,12347,537626,2010-12-07 14:57:00,711.79,2,37.00
5,12347,542237,2011-01-26 14:30:00,475.39,3,49.00
6,12347,549222,2011-04-07 10:43:00,636.25,4,70.00
7,12347,556201,2011-06-09 13:01:00,382.52,5,63.00
8,12347,562032,2011-08-02 08:48:00,584.91,6,53.00
9,12347,573511,2011-10-31 12:25:00,"1,294.32",7,90.00


In [14]:
second_orders = orders_seq[orders_seq["order_nr"] == 2]
share_repeat = second_orders["customer_id"].nunique() / orders_seq["customer_id"].nunique()
print(f"Customers with at least 2 purchases: {100 * share_repeat:.1f}%")
print("Days between 1st and 2nd purchase:")
print(second_orders["days_since_prev_order"].describe(percentiles=[0.25, 0.5, 0.75]))

Customers with at least 2 purchases: 72.4%
Days between 1st and 2nd purchase:
count   4,234.00
mean       98.54
std       113.28
min         0.00
25%        21.00
50%        56.00
75%       132.75
max       714.00
Name: days_since_prev_order, dtype: float64


## 1.8 Sanity checks
Numbers that should make business sense before we trust the data:
- date range and number of customers;
- return rate (returned value / purchased value);
- **returns without a purchase in the data**: customers whose return refers to an order placed *before* the dataset starts. Their history is incomplete (left-censoring) — we flag them for step 2.

In [15]:
purch = events[events["event_type"] == "purchase"]
rets = events[events["event_type"] == "return"]

first_purchase = purch.groupby("customer_id")["event_time"].min()
first_return = rets.groupby("customer_id")["event_time"].min()
ret_before_purch = first_return[first_return.index.map(lambda c: c not in first_purchase.index or first_return[c] < first_purchase[c])]

summary = pd.Series({
    "date_from": events["event_time"].min(),
    "date_to": events["event_time"].max(),
    "customers": events["customer_id"].nunique(),
    "customers_with_purchase": purch["customer_id"].nunique(),
    "purchase_invoices": len(purch),
    "return_invoices": len(rets),
    "gross_revenue": purch["value"].sum(),
    "returned_value": -rets["value"].sum(),
    "return_rate_pct": 100 * -rets["value"].sum() / purch["value"].sum(),
    "customers_return_before_first_purchase": len(ret_before_purch),
})
summary

,0
date_from,2009-12-01 07:45:00
date_to,2011-12-09 12:50:00
customers,5875
customers_with_purchase,5852
purchase_invoices,36594
return_invoices,7282
gross_revenue,"17,068,567.97"
returned_value,"709,953.43"
return_rate_pct,4.16
customers_return_before_first_purchase,209


How skewed is customer value? This previews the core challenge of LTV: most customers are worth little, a few are worth a lot.

In [16]:
net_value = events.groupby("customer_id")["value"].sum()
print(net_value.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))
top10_share = net_value.sort_values(ascending=False).head(int(len(net_value) * 0.1)).sum() / net_value.sum()
print(f"\nTop 10% of customers = {100 * top10_share:.1f}% of net revenue")

count     5,875.00
mean      2,784.45
std      13,799.32
min      -1,343.24
10%         154.61
25%         327.80
50%         835.22
75%       2,164.28
90%       5,198.47
99%      26,428.76
max     578,408.64
Name: value, dtype: float64

Top 10% of customers = 63.3% of net revenue


## 1.9 Save for step 2

In [17]:
events.to_csv("events_clean.csv", index=False)
pd.DataFrame(log).to_csv("cleaning_log.csv", index=False)
orders_seq.to_csv("orders_sequence.csv", index=False)
print(f"Saved events_clean.csv ({len(events):,} rows) and cleaning_log.csv")

Saved events_clean.csv (43,876 rows) and cleaning_log.csv


## Step 1 — takeaways
Written automatically from the results above.

In [18]:
from IPython.display import Markdown, display

display(Markdown(f"""
1. **Data quality:** {100 * checks["missing_customer_id"].mean():.1f}% of rows had no customer ID and could not be used for LTV; {100 * checks["exact_duplicate"].mean():.1f}% were exact duplicates.
2. **Returns matter:** {summary["return_rate_pct"]:.1f}% of purchased value was returned. Ignoring returns would overstate customer value.
3. **Value is concentrated:** the top 10% of customers generate {100 * top10_share:.1f}% of net revenue, so telling high-value customers apart early is worth a lot.
4. **Repeat behaviour:** {100 * share_repeat:.1f}% of customers buy a second time; the median gap between 1st and 2nd purchase is {second_orders["days_since_prev_order"].median():.0f} days.
5. **Incomplete histories:** {len(ret_before_purch):,} customers returned items before their first recorded purchase; their history starts before the data does.

**Next step:** for each customer, build features from their first 30 days and a target = net value in the following 180 days.
"""))


1. **Data quality:** 22.8% of rows had no customer ID and could not be used for LTV; 3.2% were exact duplicates.
2. **Returns matter:** 4.2% of purchased value was returned. Ignoring returns would overstate customer value.
3. **Value is concentrated:** the top 10% of customers generate 63.3% of net revenue, so telling high-value customers apart early is worth a lot.
4. **Repeat behaviour:** 72.4% of customers buy a second time; the median gap between 1st and 2nd purchase is 56 days.
5. **Incomplete histories:** 209 customers returned items before their first recorded purchase; their history starts before the data does.

**Next step:** for each customer, build features from their first 30 days and a target = net value in the following 180 days.


# Step 2: features and target

**Goal:** for each new customer, describe what we know in their **first 30 days** (features) and measure what they are actually worth in the **following 180 days** (target).

This is the exact situation an ad platform is in: it must decide how much a new customer is worth *soon after* they arrive, long before their real value is known.

```
first purchase (day 0) ── observation window: days 0–30 ──┬── prediction window: days 30–210 ──
                          → features                       │   → target (net value)
```

**Rule we never break:** features only use data from days 0–30, the target only from days 30–210. If information from the future leaks into the features, the model looks great on paper and fails in reality (*data leakage*).

## 2.0 Data from step 1
We continue directly from the `events` table built above, so nothing needs to be uploaded. The cell reloads it from `events_clean.csv` only if it's missing (e.g. after restarting the runtime).

In [19]:
if "events" not in globals():
    events = pd.read_csv("events_clean.csv", parse_dates=["event_time"],
                         dtype={"customer_id": str, "invoice": str})
print(f"{len(events):,} events | {events['customer_id'].nunique():,} customers | "
      f"{events['event_time'].min():%Y-%m-%d} -> {events['event_time'].max():%Y-%m-%d}")

43,876 events | 5,875 customers | 2009-12-01 -> 2011-12-09


## 2.1 Parameters
All key choices in one place, so they are easy to change and test later:
- `OBS_DAYS = 30`: how long we observe a new customer before predicting.
- `PRED_DAYS = 180`: how far ahead we measure their value.
- `BURN_IN_MONTHS = 3`: customers first seen in the first 3 months are excluded, because many of them are probably *old* customers whose history started before the data (left-censoring).

In [20]:
OBS_DAYS = 30
PRED_DAYS = 180
BURN_IN_MONTHS = 3

DATA_START = events["event_time"].min()
DATA_END = events["event_time"].max()
COHORT_START = DATA_START + pd.DateOffset(months=BURN_IN_MONTHS)
COHORT_END = DATA_END - pd.Timedelta(days=OBS_DAYS + PRED_DAYS)

print(f"New customers included if first purchase is between {COHORT_START:%Y-%m-%d} and {COHORT_END:%Y-%m-%d}")

New customers included if first purchase is between 2010-03-01 and 2011-05-13


## 2.2 Build the cohort of new customers
A customer enters the analysis only if:
1. their first purchase falls **after the burn-in** (so they are genuinely new);
2. their first purchase is **early enough** that we can watch the full 30 + 180 days before the data ends;
3. they have **no return before their first purchase** (a sign their history started before the data).

Each exclusion is counted, as in step 1.

In [21]:
purchases = events[events["event_type"] == "purchase"]
returns = events[events["event_type"] == "return"]

t0 = purchases.groupby("customer_id")["event_time"].min().rename("t0")
first_return = returns.groupby("customer_id")["event_time"].min()

cust = t0.to_frame()
cust["return_before_first_purchase"] = first_return.reindex(cust.index) < cust["t0"]

cohort_log = [("customers with at least one purchase", len(cust))]
cust = cust[cust["t0"] >= COHORT_START]
cohort_log.append(("first purchase after burn-in", len(cust)))
cust = cust[cust["t0"] <= COHORT_END]
cohort_log.append(("full 30+180 days observable", len(cust)))
cust = cust[~cust["return_before_first_purchase"]]
cohort_log.append(("no return before first purchase", len(cust)))

pd.DataFrame(cohort_log, columns=["filter", "customers_left"])

,filter,customers_left
0,customers with at least one purchase,5852
1,first purchase after burn-in,4158
2,full 30+180 days observable,3162
3,no return before first purchase,3085


## 2.3 Split each customer's events into the two windows
We attach each customer's `t0` (first purchase time) to all their events and compute `days_since_t0`. Then:
- `obs` = events in days **[0, 30)** → used for features;
- `fut` = events in days **[30, 210)** → used for the target.

In [22]:
ev = events.merge(cust[["t0"]], left_on="customer_id", right_index=True, how="inner")
ev["days_since_t0"] = (ev["event_time"] - ev["t0"]).dt.total_seconds() / 86400

obs = ev[(ev["days_since_t0"] >= 0) & (ev["days_since_t0"] < OBS_DAYS)]
fut = ev[(ev["days_since_t0"] >= OBS_DAYS) & (ev["days_since_t0"] < OBS_DAYS + PRED_DAYS)]

print(f"Events in observation window: {len(obs):,} | in prediction window: {len(fut):,}")

Events in observation window: 4,499 | in prediction window: 5,163


## 2.4 Features (what we know after 30 days)

| Feature | What it captures |
|---|---|
| `first_order_value`, `first_order_items`, `first_order_lines` | Size of the first basket — **this is all an ad platform normally sees** |
| `first_order_avg_price` | Cheap vs. premium products |
| `n_orders_30d`, `revenue_30d` | Early repeat buying |
| `days_to_2nd_order` | How fast they came back (set to 30 if they didn't) |
| `n_returns_30d`, `returned_value_30d` | Early returns: a warning sign |
| `country_uk` | Domestic vs. international customer |
| `first_month`, `first_in_q4` | Seasonality: Christmas shoppers may behave differently |

In [23]:
obs_p = obs[obs["event_type"] == "purchase"].sort_values(["customer_id", "event_time", "invoice"])
obs_r = obs[obs["event_type"] == "return"]

obs_p = obs_p.assign(order_nr=obs_p.groupby("customer_id").cumcount() + 1)

first_order = obs_p[obs_p["order_nr"] == 1].set_index("customer_id")
second_order_day = obs_p[obs_p["order_nr"] == 2].set_index("customer_id")["days_since_t0"]

f = pd.DataFrame(index=cust.index)
f["first_order_value"] = first_order["value"]
f["first_order_items"] = first_order["n_items"]
f["first_order_lines"] = first_order["n_lines"]
f["first_order_avg_price"] = f["first_order_value"] / f["first_order_items"]

f["n_orders_30d"] = obs_p.groupby("customer_id").size().reindex(f.index, fill_value=0)
f["revenue_30d"] = obs_p.groupby("customer_id")["value"].sum().reindex(f.index, fill_value=0)
f["days_to_2nd_order"] = second_order_day.reindex(f.index).fillna(OBS_DAYS)

f["n_returns_30d"] = obs_r.groupby("customer_id").size().reindex(f.index, fill_value=0)
f["returned_value_30d"] = -obs_r.groupby("customer_id")["value"].sum().reindex(f.index, fill_value=0)

f["country_uk"] = (first_order["country"] == "United Kingdom").astype(int)
f["first_month"] = cust["t0"].dt.month
f["first_in_q4"] = f["first_month"].isin([10, 11, 12]).astype(int)

f.head()

,first_order_value,first_order_items,first_order_lines,first_order_avg_price,n_orders_30d,revenue_30d,days_to_2nd_order,n_returns_30d,returned_value_30d,country_uk,first_month,first_in_q4
customer_id,,,,,,,,,,,,
12346,27.05,5,5,5.41,1,27.05,30.00,0,-0.00,1,3,0
12347,611.53,509,40,1.20,1,611.53,30.00,0,-0.00,0,10,1
12348,221.16,372,19,0.59,1,221.16,30.00,0,-0.00,0,9,0
12350,294.40,196,16,1.50,1,294.40,30.00,0,-0.00,0,2,0
12351,300.93,261,21,1.15,1,300.93,30.00,0,-0.00,0,11,1


## 2.5 Target (what they are worth afterwards)
- `target_value_180d`: **net** value in days 30–210 = purchases minus returns. This is the number a pLTV model tries to predict.
- `target_any_purchase`: 1 if they bought anything at all in that period.

Customers with no events in the window get 0 — they are real zeros, not missing data.

In [24]:
fut_p = fut[fut["event_type"] == "purchase"]
f["target_value_180d"] = fut.groupby("customer_id")["value"].sum().reindex(f.index, fill_value=0)
f["target_any_purchase"] = (fut_p.groupby("customer_id").size().reindex(f.index, fill_value=0) > 0).astype(int)

f[["target_value_180d", "target_any_purchase"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])

,target_value_180d,target_any_purchase
count,"3,085.00","3,085.00"
mean,469.75,0.52
std,"1,339.69",0.50
min,"-1,700.40",0.00
25%,0.00,0.00
50%,73.70,1.00
75%,514.36,1.00
90%,"1,183.06",1.00
99%,"4,774.23",1.00
max,"37,258.18",1.00


## 2.6 Look at the target
LTV targets are typically **zero-inflated** (many customers never come back) and **right-skewed** (a few are worth a lot). This shapes how we model and evaluate in step 3.

In [25]:
y = f["target_value_180d"]
top10 = y.sort_values(ascending=False).head(max(1, len(y) // 10)).sum() / y.sum()

print(f"Customers in cohort:              {len(y):,}")
print(f"Share with zero future value:     {100 * (y <= 0).mean():.1f}%")
print(f"Share buying again (days 30-210): {100 * f['target_any_purchase'].mean():.1f}%")
print(f"Mean vs median future value:      {y.mean():,.2f} vs {y.median():,.2f}")
print(f"Top 10% of customers' share:      {100 * top10:.1f}% of future value")

Customers in cohort:              3,085
Share with zero future value:     48.5%
Share buying again (days 30-210): 51.6%
Mean vs median future value:      469.75 vs 73.70
Top 10% of customers' share:      60.0% of future value


## 2.7 Which early signals relate to future value?
We use **Spearman correlation**, which works on ranks rather than raw values. Because the target is so skewed, ordinary (Pearson) correlation would be dominated by a handful of huge customers.

In [26]:
feature_cols = [c for c in f.columns if not c.startswith("target")]
corr = (f[feature_cols].corrwith(f["target_value_180d"], method="spearman")
        .sort_values(ascending=False).rename("spearman_with_target"))
corr.to_frame()

,spearman_with_target
revenue_30d,0.30
first_order_value,0.23
n_orders_30d,0.20
first_order_items,0.20
first_order_lines,0.18
n_returns_30d,0.13
returned_value_30d,0.12
country_uk,-0.03
first_order_avg_price,-0.03
first_month,-0.16


Two simple comparisons that tell the story without any model:

In [27]:
def compare(col, labels):
    g = f.groupby(col)["target_value_180d"].agg(customers="size", avg_future_value="mean",
                                               share_buying_again=lambda s: (s > 0).mean())
    g.index = [labels[i] for i in g.index]
    return g

print("Came back within 30 days?")
display(compare(f["n_orders_30d"] >= 2, {False: "no", True: "yes"}))
print("\nReturned something within 30 days?")
display(compare(f["n_returns_30d"] > 0, {False: "no", True: "yes"}))

Came back within 30 days?


,customers,avg_future_value,share_buying_again
no,2493,333.97,0.48
yes,592,"1,041.51",0.67



Returned something within 30 days?


,customers,avg_future_value,share_buying_again
no,2576,423.53,0.49
yes,509,703.66,0.62


## 2.8 Preview: how good is the "naive" signal?
Today an ad platform values a customer by their **first order**. We sort customers by first-order value into 10 equal groups (deciles) and look at their *actual* future value. If the first order were a good signal, future value would rise steadily from decile 1 to 10. In step 3 we'll compare this with a model.

In [28]:
f["naive_decile"] = pd.qcut(f["first_order_value"].rank(method="first"), 10, labels=False) + 1
naive = f.groupby("naive_decile").agg(customers=("target_value_180d", "size"),
                                      avg_first_order=("first_order_value", "mean"),
                                      avg_future_value=("target_value_180d", "mean"))
naive

,customers,avg_first_order,avg_future_value
naive_decile,,,
1,309,70.67,151.52
2,308,124.55,209.26
3,309,162.30,237.42
4,308,204.35,302.82
5,309,252.06,356.47
6,308,303.74,376.95
7,308,351.05,404.58
8,309,436.70,540.07
9,308,602.60,648.98


## 2.9 Save for step 3

In [29]:
f.drop(columns="naive_decile").assign(t0=cust["t0"]).to_csv("customer_features.csv")
print(f"Saved customer_features.csv ({len(f):,} customers, {len(feature_cols)} features + 2 targets)")

Saved customer_features.csv (3,085 customers, 12 features + 2 targets)


## Step 2 — takeaways
Written automatically from the results above.

In [30]:
came_back = f["n_orders_30d"] >= 2
top_corr = corr.dropna().abs().sort_values(ascending=False).index[:2].tolist()
naive_top = naive["avg_future_value"].iloc[-1]
naive_rho = f[["first_order_value", "target_value_180d"]].corr(method="spearman").iloc[0, 1]

display(Markdown(f"""
1. **Cohort:** {len(f):,} new customers, first purchase between {cust["t0"].min():%d %b %Y} and {cust["t0"].max():%d %b %Y}.
2. **Zero-inflated target:** {100 * (y <= 0).mean():.1f}% bring no value in the following 180 days; the top 10% bring {100 * top10:.1f}% of future value.
3. **Strongest early signals (Spearman):** `{top_corr[0]}` and `{top_corr[1]}`.
4. **Coming back early:** customers with a 2nd order within 30 days are worth {f.loc[came_back, "target_value_180d"].mean():,.2f} on average over the next 180 days, vs {f.loc[~came_back, "target_value_180d"].mean():,.2f} for the others.
5. **Naive signal:** first-order value has a Spearman correlation of {naive_rho:.2f} with future value; the top decile by first order is worth {naive_top:,.2f} on average vs {y.mean():,.2f} overall. Step 3 tests whether a model does better.
"""))


1. **Cohort:** 3,085 new customers, first purchase between 01 Mar 2010 and 12 May 2011.
2. **Zero-inflated target:** 48.5% bring no value in the following 180 days; the top 10% bring 60.0% of future value.
3. **Strongest early signals (Spearman):** `revenue_30d` and `first_order_value`.
4. **Coming back early:** customers with a 2nd order within 30 days are worth 1,041.51 on average over the next 180 days, vs 333.97 for the others.
5. **Naive signal:** first-order value has a Spearman correlation of 0.23 with future value; the top decile by first order is worth 1,468.09 on average vs 469.75 overall. Step 3 tests whether a model does better.


# Step 3: the pLTV model vs. the naive signal

**Question:** can we rank new customers by future value better than the first order does?

We compare three ways of scoring each customer after 30 days:
1. **Naive signal** — first-order value. This is roughly what an ad platform sees today.
2. **Linear regression** — a simple, transparent model on all 12 features.
3. **Gradient boosting** — a stronger tree-based model that captures non-linear patterns and interactions.

**What "better" means here.** An ad platform uses the value to decide *whom to bid more for*. So what matters most is **ranking**: are the customers we score highest really the most valuable? We also check **calibration** (are predicted values the right size on average?), because the platform uses the number itself when bidding.

## 3.0 Setup
`scikit-learn` is the standard Python library for machine learning. We import the two models and two helper functions.

In [31]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy.stats import spearmanr

data = f.drop(columns="naive_decile", errors="ignore").copy()
data["t0"] = cust["t0"]
feature_cols = [c for c in data.columns if not c.startswith("target") and c != "t0"]
print(len(data), "customers |", len(feature_cols), "features:", feature_cols)

3085 customers | 12 features: ['first_order_value', 'first_order_items', 'first_order_lines', 'first_order_avg_price', 'n_orders_30d', 'revenue_30d', 'days_to_2nd_order', 'n_returns_30d', 'returned_value_30d', 'country_uk', 'first_month', 'first_in_q4']


## 3.1 Out-of-time split
We train on customers who arrived **earlier** and test on customers who arrived **later**. A random split would mix periods and look better than reality; in production a model is always trained on the past and used on the future.

We use the earliest ~70% of customers (by first-purchase date) for training and the latest ~30% for testing.

In [32]:
split_date = data["t0"].quantile(0.7)
train = data[data["t0"] < split_date]
test = data[data["t0"] >= split_date].copy()

print(f"Split date: {split_date:%Y-%m-%d}")
print(f"Train: {len(train):,} customers | Test: {len(test):,} customers")
print(f"Avg future value — train: {train['target_value_180d'].mean():,.2f} | test: {test['target_value_180d'].mean():,.2f}")

Split date: 2010-11-01
Train: 2,159 customers | Test: 926 customers
Avg future value — train: 509.80 | test: 376.35


## 3.2 Train the models
- **Target transformation for the linear model.** The target is very skewed, so we fit the linear model on `log(1 + value)` and transform predictions back. Without this, a few huge customers would dominate the fit.
- **Negative targets** (net returners) are set to 0 for training: from an advertiser's point of view, both "worth nothing" and "worth less than nothing" mean *don't pay to acquire them*. Evaluation still uses the real net values.
- **Poisson loss for gradient boosting.** It is designed for non-negative, skewed targets with many zeros — the shape of LTV.

In [33]:
X_train, X_test = train[feature_cols], test[feature_cols]
y_train = train["target_value_180d"].clip(lower=0)

lin = LinearRegression().fit(X_train, np.log1p(y_train))
test["pred_linear"] = np.expm1(lin.predict(X_test)).clip(min=0)

gbm = HistGradientBoostingRegressor(loss="poisson", max_iter=300, learning_rate=0.05,
                                    max_leaf_nodes=15, min_samples_leaf=40, random_state=42)
gbm.fit(X_train, y_train)
test["pred_gbm"] = gbm.predict(X_test)

test["pred_naive"] = test["first_order_value"]
test[["first_order_value", "pred_linear", "pred_gbm", "target_value_180d"]].head()

,first_order_value,pred_linear,pred_gbm,target_value_180d
customer_id,,,,
12350,294.40,46.69,417.22,0.00
12351,300.93,6.95,143.26,0.00
12352,143.75,9.85,298.76,521.18
12354,"1,079.40",89.17,587.60,0.00
12357,"11,229.99","1,778.78","1,612.06",0.00


## 3.3 Ranking quality: decile tables
For each score we sort test customers into 10 equal groups, from lowest score (decile 1) to highest (decile 10), and show their **actual** average future value.

A good score shows actual value rising steadily from decile 1 to 10, with a big gap at the top. A useless score shows a flat line.

In [34]:
scores = {"naive": "pred_naive", "linear": "pred_linear", "gbm": "pred_gbm"}

def decile_table(df, score_col):
    d = pd.qcut(df[score_col].rank(method="first"), 10, labels=False) + 1
    return df.groupby(d)["target_value_180d"].mean()

deciles = pd.DataFrame({name: decile_table(test, col) for name, col in scores.items()})
deciles.index.name = "score_decile"
deciles

,naive,linear,gbm
score_decile,,,
1,151.48,71.66,162.98
2,187.43,105.45,98.69
3,167.95,177.07,144.55
4,260.60,233.75,153.30
5,226.77,427.43,283.12
6,298.45,216.37,205.34
7,392.55,313.10,279.26
8,397.32,316.73,386.99
9,678.01,395.08,688.23


## 3.4 Summary metrics
- **Spearman correlation** between score and actual value: overall ranking quality (1 = perfect order, 0 = random).
- **Top-10% / top-20% capture**: share of all future value held by the customers the score ranks in the top 10% / 20%. If you could only acquire those customers, how much of the value would you get? Random scoring gives 10% / 20%.
- **Top-decile lift**: average value in the top decile divided by the overall average. 3× means the top group is worth three times an average customer.

In [35]:
def capture(df, score_col, share):
    n = max(1, int(len(df) * share))
    top = df.sort_values(score_col, ascending=False).head(n)
    return top["target_value_180d"].sum() / df["target_value_180d"].sum()

overall_mean = test["target_value_180d"].mean()
metrics = pd.DataFrame({
    name: {
        "spearman": spearmanr(test[col], test["target_value_180d"])[0],
        "top10_capture_%": 100 * capture(test, col, 0.10),
        "top20_capture_%": 100 * capture(test, col, 0.20),
        "top_decile_lift_x": deciles[name].loc[10] / overall_mean,
    } for name, col in scores.items()
}).T
metrics

,spearman,top10_capture_%,top20_capture_%,top_decile_lift_x
naive,0.18,26.51,44.65,2.66
linear,0.31,40.08,50.59,4.00
gbm,0.32,36.29,54.46,3.61


## 3.5 Calibration of the best model
Ranking is not enough: the platform bids with the predicted *amount*. We compare the average predicted value with the average actual value in each decile of the gradient boosting model. A ratio near 1 means the model is well calibrated; above 1 it under-predicts, below 1 it over-predicts.

In [36]:
d = pd.qcut(test["pred_gbm"].rank(method="first"), 10, labels=False) + 1
calib = test.groupby(d).agg(avg_predicted=("pred_gbm", "mean"),
                            avg_actual=("target_value_180d", "mean"))
calib["actual_over_predicted"] = calib["avg_actual"] / calib["avg_predicted"]
calib.index.name = "gbm_decile"
calib

,avg_predicted,avg_actual,actual_over_predicted
gbm_decile,,,
1,53.69,162.98,3.04
2,103.68,98.69,0.95
3,142.83,144.55,1.01
4,180.50,153.30,0.85
5,228.33,283.12,1.24
6,297.18,205.34,0.69
7,390.69,279.26,0.71
8,507.45,386.99,0.76
9,736.43,688.23,0.93


## 3.6 Which signals does the model use?
**Permutation importance:** we shuffle one feature at a time in the test set and measure how much the model's ranking gets worse. A big drop means the model relies on that feature. It's model-agnostic and measured on unseen data.

In [37]:
from sklearn.inspection import permutation_importance

def spearman_scorer(model, X, y):
    return spearmanr(model.predict(X), y)[0]

imp = permutation_importance(gbm, X_test, test["target_value_180d"], scoring=spearman_scorer,
                             n_repeats=10, random_state=42)
importance = (pd.Series(imp.importances_mean, index=feature_cols)
              .sort_values(ascending=False).rename("drop_in_spearman"))
importance.to_frame()

,drop_in_spearman
revenue_30d,0.21
first_month,0.11
first_order_lines,0.01
first_order_items,0.01
n_orders_30d,0.01
days_to_2nd_order,0.01
first_order_value,0.01
returned_value_30d,0.00
first_order_avg_price,0.00
country_uk,0.00


# Step 4: how would we prove it works for a client?

A better ranking on historical data is promising, but it does not prove that *sending this signal to Meta or Google* increases revenue. That needs an experiment with a control group. This step designs that experiment and sizes it using the real variability of customer value in this dataset.

## 4.1 The experiment design
**Recommended: a geo experiment.**
- Split comparable regions into **treatment** (campaigns optimise on the pLTV signal) and **control** (campaigns keep the current signal).
- Check that treatment and control moved in parallel *before* the test (parallel trends).
- Run for several weeks, beyond the platform's learning phase.
- Estimate the effect with **difference-in-differences**: (change in treatment) − (change in control).

**Why not a simple A/B test between two campaigns?** Both campaigns would compete in the same auctions and reach overlapping audiences, so the groups contaminate each other.

**Main metric:** net revenue per new customer after 180 days (or a validated early proxy), not first-week ROAS — the pLTV signal can look worse on short-term ROAS while bringing better customers.

## 4.2 How variable is customer value?
The size of an experiment depends on how noisy the metric is. Customer value is extremely noisy: many zeros and a few very large values. The **coefficient of variation** (standard deviation ÷ mean) summarises this: the higher it is, the more customers we need.

In [38]:
y_all = data["target_value_180d"]
mu, sigma = y_all.mean(), y_all.std()
print(f"Mean future value per customer: {mu:,.2f}")
print(f"Standard deviation:            {sigma:,.2f}")
print(f"Coefficient of variation:      {sigma / mu:,.1f}")

Mean future value per customer: 469.75
Standard deviation:            1,339.69
Coefficient of variation:      2.9


## 4.3 Power calculation: how many customers do we need?
To detect a difference δ between two groups with **80% power** and **5% significance**, a standard approximation for the number of customers *per group* is:

$$n \approx \frac{2\,(z_{0.975} + z_{0.80})^2\,\sigma^2}{\delta^2} \approx \frac{16\,\sigma^2}{\delta^2}$$

- **Power 80%**: if the effect is real, the test detects it 8 times out of 10.
- **Significance 5%**: if there is no effect, we wrongly claim one only 5% of the time.
- δ is the effect we want to detect, here a % lift on the mean value.

In [39]:
from scipy.stats import norm

z = norm.ppf(0.975) + norm.ppf(0.80)
lifts = [0.05, 0.10, 0.15, 0.20, 0.30]
power = pd.DataFrame({
    "lift_%": [int(100 * l) for l in lifts],
    "delta_per_customer": [mu * l for l in lifts],
    "customers_per_group": [int(np.ceil(2 * z**2 * sigma**2 / (mu * l)**2)) for l in lifts],
})
power["total_customers"] = 2 * power["customers_per_group"]
power

,lift_%,delta_per_customer,customers_per_group,total_customers
0,5,23.49,51072,102144
1,10,46.97,12768,25536
2,15,70.46,5675,11350
3,20,93.95,3192,6384
4,30,140.92,1419,2838


## 4.4 How long would the test need to run?
We translate the sample size into time, using how many new customers this business acquires per week. In a real client we would use their actual acquisition volume.

In [40]:
new_per_week = data.set_index("t0").resample("W").size().mean()
power["weeks_needed"] = (power["total_customers"] / new_per_week).round(1)
print(f"New customers per week (average): {new_per_week:,.0f}")
power[["lift_%", "total_customers", "weeks_needed"]]

New customers per week (average): 49


,lift_%,total_customers,weeks_needed
0,5,102144,"2,085.90"
1,10,25536,521.50
2,15,11350,231.80
3,20,6384,130.40
4,30,2838,58.00


**How to read this.** If the table shows that detecting a 10% lift needs more weeks than the client is willing to wait, we have options:
- test for a larger, realistic effect only;
- use a **less noisy metric** (e.g. revenue capped at the 99th percentile, or "bought again: yes/no");
- use **variance reduction** such as CUPED, which adjusts for pre-test behaviour;
- accept an early proxy (predicted value) and validate it on real 180-day value later.

Being explicit about this *before* launch avoids a common failure: an underpowered test that "shows no effect" when there actually was one.

# Summary of results
This cell writes the summary automatically from the numbers computed above, so it always matches your run.

In [41]:
from IPython.display import Markdown, display

usable_pct = 100 * len(df) / len(raw)
missing_id_pct = 100 * checks["missing_customer_id"].mean()
return_pct = summary["return_rate_pct"]
repeat_pct = 100 * share_repeat
median_gap = second_orders["days_since_prev_order"].median()

zero_pct = 100 * (y <= 0).mean()
top10_pct = 100 * top10

cap = metrics["top20_capture_%"]
best = cap.drop("naive").idxmax()
best_name = {"linear": "linear regression", "gbm": "gradient boosting"}[best]
top_feats = importance.index[:2].tolist()

row10 = power.loc[power["lift_%"] == 10].iloc[0]
row20 = power.loc[power["lift_%"] == 20].iloc[0]

verdict = ("beats" if cap[best] > cap["naive"] + 2 else
           "does not clearly beat")

text = f"""
**1. Data quality.** Only {usable_pct:.1f}% of raw rows were usable for LTV; the main loss is missing customer IDs ({missing_id_pct:.1f}% of rows). {return_pct:.1f}% of purchased value was returned, so ignoring returns would overstate customer value.

**2. Repeat behaviour.** {repeat_pct:.1f}% of customers buy at least twice; the median gap between the 1st and 2nd purchase is {median_gap:.0f} days.

**3. Target.** In the cohort of {len(data):,} new customers, {zero_pct:.1f}% bring no value in the following 180 days, while the top 10% bring {top10_pct:.1f}% of all future value.

**4. Naive signal vs model.** Ranking customers by first-order value, the top 20% hold {cap["naive"]:.1f}% of future value. With {best_name}, the top 20% hold {cap[best]:.1f}% (Spearman {metrics.loc[best, "spearman"]:.2f} vs {metrics.loc["naive", "spearman"]:.2f} for the first order). The model **{verdict}** the naive signal.

**5. Key signals.** The model relies most on `{top_feats[0]}` and `{top_feats[1]}`.

**6. Measurement.** Customer value is very noisy (coefficient of variation {sigma / mu:.1f}). Detecting a +10% lift with 80% power needs about {row10["total_customers"]:,.0f} new customers in total ({row10["weeks_needed"]:,.0f} weeks at this retailer's acquisition rate); a +20% lift needs {row20["total_customers"]:,.0f} ({row20["weeks_needed"]:,.0f} weeks). Recommended design: a geo experiment estimated with difference-in-differences.
"""
display(Markdown(text))


**1. Data quality.** Only 74.4% of raw rows were usable for LTV; the main loss is missing customer IDs (22.8% of rows). 4.2% of purchased value was returned, so ignoring returns would overstate customer value.

**2. Repeat behaviour.** 72.4% of customers buy at least twice; the median gap between the 1st and 2nd purchase is 56 days.

**3. Target.** In the cohort of 3,085 new customers, 48.5% bring no value in the following 180 days, while the top 10% bring 60.0% of all future value.

**4. Naive signal vs model.** Ranking customers by first-order value, the top 20% hold 44.7% of future value. With gradient boosting, the top 20% hold 54.5% (Spearman 0.32 vs 0.18 for the first order). The model **beats** the naive signal.

**5. Key signals.** The model relies most on `revenue_30d` and `first_month`.

**6. Measurement.** Customer value is very noisy (coefficient of variation 2.9). Detecting a +10% lift with 80% power needs about 25,536 new customers in total (522 weeks at this retailer's acquisition rate); a +20% lift needs 6,384 (130 weeks). Recommended design: a geo experiment estimated with difference-in-differences.


# Limitations
- One retailer, mostly UK, many wholesale buyers: not a typical consumer e-commerce business.
- No acquisition-channel or ad data: we predict value, but cannot simulate how Meta or Google would react to the signal.
- Simple features and models, with no tuning or causal methods.
- Good ranking on historical data is not proof of incremental revenue: that needs the experiment in step 4.